# Building a Retrieval-Augmented Generation (RAG) Pipeline

This notebook walks through a simple implementation of a
retrieval-augmented generation (RAG) system. RAG combines the
strengths of information retrieval and large language models (LLMs)
to answer questions grounded in external knowledge. The seven
components of a RAG pipeline, as described in our chapter, are:

1. **Indexing** – convert a corpus into a searchable index.
2. **Query construction** – rewrite or expand questions into one or
   more search queries.
3. **Candidate retrieval** – retrieve potentially relevant passages
   from the index.
4. **Re-ranking** – reorder the retrieved candidates using a more
   accurate similarity model.
5. **Context assembly** – select a small set of passages to provide
   to the generator.
6. **Grounded generation** – generate an answer conditioned on the
   retrieved context (we use a simple placeholder here).
7. **Attribution and logging** – keep track of which sources were
   used to support the answer.

In this tutorial we construct a toy corpus, build a TF-IDF
retriever and re-ranker with scikit-learn, and assemble a context
to produce an answer. Finally, we compute some basic evaluation
metrics (Recall@k, Precision@k, MRR and nDCG) to quantify
retrieval performance. Where a real system would call an LLM to
rewrite queries or generate answers, we include placeholders and
comments indicating where those components would fit.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Define a small toy corpus. In a real system this would be a large
# collection of documents, possibly split into overlapping chunks. Each
# entry here contains a short paragraph of text and an identifier for
# logging.
documents = [
    {"id": "doc1", "text": "The Eiffel Tower is located in Paris and is one of the most famous landmarks in the world. It stands on the Champ de Mars near the Seine River."},
    {"id": "doc2", "text": "Paris is the capital of France. It is known for its museums, including the Louvre, and for its cuisine and fashion."},
    {"id": "doc3", "text": "Berlin is the capital of Germany. The city has a rich history and is known for the Brandenburg Gate."},
    {"id": "doc4", "text": "Retrieval-augmented generation (RAG) systems combine information retrieval with language models to provide grounded answers. They first retrieve documents and then condition the generation on those documents."},
    {"id": "doc5", "text": "Graph-RAG extends RAG to knowledge graphs, allowing multi-hop reasoning over linked entities. Code-RAG applies similar principles to source code."},
    {"id": "doc6", "text": "Multi-modal RAG enables retrieval of images and other modalities, for example using CLIP to find visual evidence given a textual query."},
]

corpus = [doc["text"] for doc in documents]
doc_ids = [doc["id"] for doc in documents]

## 1. Indexing

The first step in a RAG pipeline is to convert the raw source
documents into searchable representations. Here we use a TF-IDF
vectorizer to build sparse representations that capture exact
term matches. We also build a low-dimensional latent semantic
index via truncated singular value decomposition (SVD) to enable
coarse semantic re-ranking.

In [ ]:
# Create a TF-IDF vectorizer and fit it on the corpus
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# Build a latent semantic index (LSI) for semantic re-ranking
svd = TruncatedSVD(n_components=100, random_state=42)
lsi_matrix = svd.fit_transform(tfidf_matrix)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"LSI matrix shape: {lsi_matrix.shape}")

## 2. Query construction

In a fully fledged system, queries may be rewritten, expanded or
decomposed into sub-queries. For example, the HyDE technique
generates a hypothetical answer and uses it to retrieve supporting
documents. In this notebook we keep things simple and just
return the user question as a single query. Placeholders show
where you could add LLM-based rewrite logic.

In [ ]:
from typing import List, Tuple

def rewrite_question(question: str, n_variants: int = 1) -> List[str]:
    # Return a list of search queries derived from the question.
    # This placeholder simply returns the original question. In practice
    # you could call an LLM to generate paraphrases, expand acronyms or
    # produce a hypothetical answer (HyDE) and append it to the list.
    return [question]

## 3. Candidate retrieval

We implement a simple function `retrieve` that embeds the queries
using our TF-IDF vectorizer and performs cosine similarity search
against the indexed documents. The function returns the top
`k_candidates` document indices along with their similarity
scores.

In [ ]:
def retrieve(index_matrix, query: str, k_candidates: int = 3) -> List[Tuple[int, float]]:
    # Retrieve candidate documents for a query using cosine similarity.
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, index_matrix)[0]
    # Get indices of top candidates
    top_idx = np.argsort(sims)[::-1][:k_candidates]
    return [(idx, sims[idx]) for idx in top_idx]

## 4. Re-ranking

After retrieving a pool of candidates, we re-rank them using a
latent semantic index built via truncated SVD. This provides a
coarse semantic similarity measure that may surface documents
containing paraphrased content even if they share few exact
terms. In more advanced systems, cross-encoders or dual-encoders
provide this refinement.

In [ ]:
def rerank(candidates: List[Tuple[int, float]], query: str, k_top: int = 2) -> List[Tuple[int, float]]:
    # Re-rank candidate documents using a latent semantic index.
    # Compute query representation in LSI space
    query_tfidf = vectorizer.transform([query])
    query_lsi = svd.transform(query_tfidf)
    scores = []
    for idx, _ in candidates:
        doc_lsi = lsi_matrix[idx].reshape(1, -1)
        sim = cosine_similarity(query_lsi, doc_lsi)[0, 0]
        scores.append((idx, sim))
    # Sort by the new scores
    reranked = sorted(scores, key=lambda x: x[1], reverse=True)
    return reranked[:k_top]

## 5. Context assembly

The re-ranking stage yields a small set of highly relevant
documents. We assemble their text into a single context string and
include metadata (e.g. document IDs) for attribution. In a real
system you may want to deduplicate overlapping passages or
enforce diversity across sources.

In [ ]:
def assemble_context(reranked_docs: List[Tuple[int, float]]) -> Tuple[str, List[str]]:
    # Assemble the text of the selected passages and record their IDs.
    context_parts = []
    used_ids = []
    for idx, score in reranked_docs:
        context_parts.append(documents[idx]["text"])
        used_ids.append(documents[idx]["id"])
    context = "

".join(context_parts)
    return context, used_ids

## 6. Grounded generation

In a full RAG system, the context assembled in the previous step
would be sent to a language model together with the original
question to generate an answer. The model would be instructed to
use information from the context and cite sources appropriately.
Here we implement a very simple generation function that
extracts the most relevant sentence from the context. If you
have access to an LLM API (such as OpenAI's `gpt-3.5-turbo`),
you could replace this function with a call to that API.

In [ ]:
import re

def generate_answer(question: str, context: str) -> str:
    # Generate a simple answer by selecting the most relevant sentence.
    # This placeholder splits the context into sentences and returns the
    # sentence with the highest word overlap with the question. This is a
    # naive proxy for grounded generation. For a real system, replace
    # this implementation with a call to your preferred LLM, passing
    # both the question and context as input.
    # Split context into sentences
    sentences = re.split(r"(?<=[.!?]) +", context)
    question_terms = set(re.findall(r"\w+", question.lower()))
    best_sent = ""
    best_score = 0
    for sent in sentences:
        sent_terms = set(re.findall(r"\w+", sent.lower()))
        overlap = len(question_terms & sent_terms)
        if overlap > best_score:
            best_score = overlap
            best_sent = sent
    return best_sent if best_sent else "I'm not sure how to answer that question."

## 7. Attribution and logging

RAG systems should log every query, the retrieved documents, and
the final answer with citations. This enables debugging and
transparency. We define a simple logger that collects this
information in a list. You can extend this to write to files or
databases.

In [ ]:
from datetime import datetime

log = []  # global list to hold logs

def log_interaction(question: str, context_ids: List[str], answer: str) -> None:
    # Record the query, context IDs and answer with a timestamp.
    log_entry = {
        "timestamp": datetime.utcnow().isoformat(),
        "question": question,
        "sources": context_ids,
        "answer": answer,
    }
    log.append(log_entry)

## Running the RAG pipeline

With all components defined, we can now wire them together in
a single function `rag_answer` that follows the pseudocode from
the chapter. It performs query rewriting, candidate retrieval,
re-ranking, context assembly, answer generation and logging.

In [ ]:
def rag_answer(question: str, k_candidates: int = 4, k_top: int = 2) -> str:
    # Execute the full RAG pipeline for a user question.
    # 2. Query construction
    queries = rewrite_question(question)
    # 3. Candidate retrieval (collect from all queries)
    candidate_pool = []
    for q in queries:
        candidate_pool.extend(retrieve(tfidf_matrix, q, k_candidates))
    # Remove duplicate candidates (keep highest score per doc)
    candidate_dict = {}
    for idx, score in candidate_pool:
        if idx not in candidate_dict or score > candidate_dict[idx]:
            candidate_dict[idx] = score
    candidates = list(candidate_dict.items())
    # 4. Re-ranking
    reranked_docs = rerank(candidates, question, k_top=k_top)
    # 5. Context assembly
    context, context_ids = assemble_context(reranked_docs)
    # 6. Generation
    answer = generate_answer(question, context)
    # 7. Logging
    log_interaction(question, context_ids, answer)
    return answer

### Example usage

Let's ask a few questions and see what answers our simple RAG
pipeline produces. You can experiment by changing the question
and seeing which documents are retrieved and used to answer it.

In [ ]:
questions = [
    "What is the capital of France?",
    "Where is the Eiffel Tower located?",
    "What does retrieval augmented generation do?",
]
for q in questions:
    print(f"
Question: {q}")
    ans = rag_answer(q)
    print(f"Answer: {ans}")

## Evaluation metrics

To assess the quality of our retrieval component, we can compute
some standard information retrieval metrics. Suppose we have
ground-truth labels indicating which documents are relevant for
each query. We then compute Recall@k, Precision@k, Mean
Reciprocal Rank (MRR), and normalized Discounted Cumulative
Gain (nDCG). The formulas are included below for reference.

\(
\mathrm{Recall@}k = rac{|\{	ext{relevant}\} \cap \{	ext{top-}k 	ext{ retrieved}\}|}{|\{	ext{relevant}\}|}
\)

\(
\mathrm{Precision@}k = rac{|\{	ext{relevant}\} \cap \{	ext{top-}k 	ext{ retrieved}\}|}{k}
\)

\(
\mathrm{MRR} = rac{1}{|Q|} \sum_{i=1}^{|Q|} rac{1}{\mathrm{rank}_i}
\)

\(
\mathrm{nDCG@}k = rac{\mathrm{DCG@}k}{\mathrm{IDCG@}k}, \quad \mathrm{DCG@}k = \sum_{i=1}^{k} rac{2^{\mathrm{rel}_i} - 1}{\log_2(i+1)}
\)

In [ ]:
def eval_metrics(question: str, relevant_ids: List[str], k: int = 3):
    # Compute retrieval metrics for a single query.
    # Retrieve and re-rank
    candidates = retrieve(tfidf_matrix, question, k_candidates=k)
    reranked_docs = rerank(candidates, question, k_top=k)
    retrieved_ids = [documents[idx]["id"] for idx, score in reranked_docs]
    # Compute recall and precision
    relevant_set = set(relevant_ids)
    retrieved_set = set(retrieved_ids)
    recall = len(relevant_set & retrieved_set) / len(relevant_set) if relevant_set else 0
    precision = len(relevant_set & retrieved_set) / k if k > 0 else 0
    # Compute MRR
    rank = next((i + 1 for i, doc_id in enumerate(retrieved_ids) if doc_id in relevant_set), None)
    mrr = (1 / rank) if rank is not None else 0
    # Compute nDCG
    # Assign relevance grades: 1 for relevant, 0 for non-relevant
    rel_grades = [1 if doc_id in relevant_set else 0 for doc_id in retrieved_ids]
    dcg = sum([(2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(rel_grades)])
    # Ideal DCG: all relevant documents at the top
    ideal_rel = sorted(rel_grades, reverse=True)
    idcg = sum([(2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rel)]) or 1
    ndcg = dcg / idcg
    return {
        "recall@k": recall,
        "precision@k": precision,
        "mrr": mrr,
        "ndcg@k": ndcg,
    }

# Example evaluation: assume for each question we know which docs are relevant
ground_truth = {
    "What is the capital of France?": ["doc2"],
    "Where is the Eiffel Tower located?": ["doc1"],
    "What does retrieval augmented generation do?": ["doc4"],
}
for q, rel_ids in ground_truth.items():
    metrics = eval_metrics(q, rel_ids, k=2)
    print(f"
Metrics for '{q}':")
    for key, value in metrics.items():
        print(f"  {key}: {value:.3f}")

## Inspecting the log

The log captures every interaction with the system. You can view
it as a Pandas DataFrame to see the questions, chosen sources and
answers. This transparency is critical for real deployments.

In [ ]:
log_df = pd.DataFrame(log)
log_df